# Prepare context safe classification data

Preserve taxonomy labels in parentheses; split by normalized event with seed 123.


In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import csv, json, ast, hashlib, re, random, os

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'data/forensic_atomic.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the forensic_atomic repository.')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CSV_PATH = ROOT / 'data/forensic_atomic.csv'
EXPERIMENT = ROOT / 'models/phi4_mini/classification/context_safe'
SPLITS_CLEAN = EXPERIMENT / 'splits'
SEED_CLEAN = 123
CLEAN_METHOD = 'kaggle_extended_taxonomy_parenthetical_safe_v3'

assert CSV_PATH.is_file(), f'CSV not found: {CSV_PATH}'
assert EXPERIMENT.resolve() != ROOT.resolve()
print('Source CSV:', CSV_PATH)
print('New experiment:', EXPERIMENT)

def sha256_file(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
RELATIONS = ['xIntent','xNeed','xAttr','xEffect','xReact','xWant',
             'oEffect','oReact','oWant','isAfter','HasSubEvent','isBefore',
             'HinderedBy','Causes','xReason','isFilledBy']
NEEDED = ['event', 'brief_context', 'crime_category', 'crime_subcategory'] + RELATIONS

def parse_relation(value, lineno, col):
    s = str(value or '').strip()
    if not s or s.lower() in ('null', 'nan', 'none'):
        return []
    try:
        value = json.loads(s)
    except json.JSONDecodeError:
        try:
            value = ast.literal_eval(s)
        except Exception as ex:
            raise ValueError(f'Row {lineno}: invalid list in {col}: {s[:100]!r}') from ex
    if not isinstance(value, list):
        raise ValueError(f'Row {lineno}: {col} is not a list')
    return [x.strip() for x in value if isinstance(x, str) and x.strip()]

rows_raw = []
with CSV_PATH.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    missing = set(NEEDED) - set(reader.fieldnames or [])
    assert not missing, f'Missing columns: {sorted(missing)}'
    for lineno, r in enumerate(reader, 2):
        row = {col: str(r.get(col, '') or '').strip() for col in NEEDED[:4]}
        if not row['event'] or not row['crime_category'] or not row['crime_subcategory']:
            continue
        row['original_row_id'] = str(r.get('original_row_id') or lineno - 1)
        for rel in RELATIONS:
            row[rel] = parse_relation(r.get(rel), lineno, rel)
        rows_raw.append(row)

assert rows_raw, 'CSV has no valid records'
ids = [r['original_row_id'] for r in rows_raw]
assert len(set(ids)) == len(ids), 'original_row_id is not unique; check the CSV'
print('Examples:', len(rows_raw))
print('Categories:', dict(Counter(r['crime_category'] for r in rows_raw)))


## Clean context and taxonomy


In [ ]:
categories_original = [

    "Violent Crimes", "Property Crimes", "Financial Crimes", "Cyber Crimes", "Organized Crimes", "Public Order / Justice Crimes",

    "Assault", "Homicide", "Kidnapping", "Robbery", "Sexual Offense", "Theft", "Burglary", "Arson", "Vandalism", "Trespassing",
    "Fraud", "Money Laundering", "Bribery", "Tax Evasion", "Hacking", "Phishing", "Identity Theft", "Online Harassment",
    "Drug Trafficking", "Racketeering", "Weapon Smuggling", "Perjury", "Obstruction of Justice", "Disorderly Conduct"
]



labels_csv = (
    {row['crime_category'] for row in rows_raw}
    | {row['crime_subcategory'] for row in rows_raw}
)
extra_labels = sorted(labels_csv - set(categories_original), key=str.casefold)
categories = categories_original + extra_labels



taxonomy_patterns = [rf"\b{re.escape(cat)}\b" for cat in sorted(categories, key=len, reverse=True)]
taxonomy_regex = re.compile("|".join(taxonomy_patterns), flags=re.IGNORECASE)


def enclosing_parenthesis(text, pos):
    """Return the outermost parenthesis pair containing the position, if any."""
    stack, matching = [], []
    for i, c in enumerate(text):
        if c == "(":
            stack.append(i)
        elif c == ")" and stack:
            opening = stack.pop()
            if opening < pos < i:
                matching.append((opening, i + 1))
    return min(matching, key=lambda pair: pair[0]) if matching else None


def previous_valid_delimiter(text, stop):
    """Return the last sentence boundary before stop, outside parentheses and abbreviations."""
    protected = set()
    abbreviations = r"\b(?:e\.g\.|i\.e\.|etc\.|dr\.|mr\.|ms\.|prof\.)"
    for match in re.finditer(abbreviations, text, flags=re.IGNORECASE):
        protected.update(i for i in range(match.start(), match.end()) if text[i] == ".")

    depth, last = 0, None
    for i, c in enumerate(text[:stop]):
        if c == "(":
            depth += 1
        elif c == ")":
            depth = max(0, depth - 1)
        elif depth == 0 and c in ".," and i not in protected:
            if c == "." and i > 0 and i + 1 < len(text):
                if text[i - 1].isdigit() and text[i + 1].isdigit():
                    continue
            last = (i, c)
    return last


def clean_brief_context(text):
    if text is None:
        return text
    text = str(text)



    while True:
        matches = list(taxonomy_regex.finditer(text))
        inside = next(
            ((m, enclosing_parenthesis(text, m.start())) for m in matches
             if enclosing_parenthesis(text, m.start()) is not None),
            None
        )
        if inside is None:
            break
        _, (start, end) = inside
        text = (text[:start].rstrip() + " " + text[end:].lstrip()).strip()
        text = re.sub(r"\s+([.,;:!?])", r"\1", text)
        text = re.sub(r"[ \t]{2,}", " ", text)



    match = taxonomy_regex.search(text)
    if match is None:
        return text
    delimiter = previous_valid_delimiter(text, match.start())
    if delimiter is None:
        return text
    idx, symbol = delimiter
    if symbol == ".":
        return text[:idx + 1].rstrip()
    return text[:idx].rstrip() + "."



problematic = ("PersonX uses stolen identities, forged documents, or manipulated "
               "financial data (e.g., synthetic identity fraud, mortgage fraud) "
               "to illegally obtain financing and purchase a house they are not entitled to.")
expected = ("PersonX uses stolen identities, forged documents, or manipulated "
            "financial data to illegally obtain financing and purchase a house they are not entitled to.")
assert clean_brief_context(problematic) == expected, clean_brief_context(problematic)

rows_clean = []
changes = []
for row in rows_raw:
    updated = dict(row)
    updated['brief_context'] = clean_brief_context(row['brief_context'])
    rows_clean.append(updated)
    if updated['brief_context'] != row['brief_context']:
        changes.append({
            'original_row_id': row['original_row_id'],
            'original': row['brief_context'],
            'cleaned': updated['brief_context']
        })

still_mentions = [r for r in rows_clean if taxonomy_regex.search(r['brief_context'])]
print('Added labels:', extra_labels)
print(f'Searched labels: {len(categories)}')
print(f'Changed contexts: {len(changes)}/{len(rows_clean)} ({len(changes)/len(rows_clean):.1%})')
print(f'Contexts still containing taxonomy terms: {len(still_mentions)}/{len(rows_clean)}')
for example in changes[:3]:
    print('\nID:', example['original_row_id'])
    print('BEFORE:', example['original'][:300])
    print('AFTER:', example['cleaned'][:300])


assert clean_brief_context('PersonX enters a building. This constitutes Burglary.') == 'PersonX enters a building.'
assert clean_brief_context('PersonX commits Fraud without earlier punctuation') == 'PersonX commits Fraud without earlier punctuation'
assert clean_brief_context('An event, involving Assault') == 'An event.'
if 'Embezzlement' in categories:
    assert clean_brief_context('A complex transaction. This is Embezzlement.') == 'A complex transaction.'
if 'Human Trafficking' in categories:
    assert clean_brief_context('PersonX recruits others, linked to Human Trafficking.') == 'PersonX recruits others.'


## Grouped 70/15/15 split


In [ ]:
def event_key(event):
    return ' '.join(event.casefold().split())

groups = defaultdict(list)
for row in rows_clean:
    groups[event_key(row['event'])].append(row)
assert len(groups) >= 10
all_pairs = {(r['crime_category'], r['crime_subcategory']) for r in rows_clean}
expected_test = len(rows_clean) * 0.15
expected_val = len(rows_clean) * 0.15
best = None
for attempt in range(300):
    keys = list(groups)
    random.Random(SEED_CLEAN + attempt).shuffle(keys)
    parts = {'train': [], 'val': [], 'test': []}
    counts = {'train': 0, 'val': 0, 'test': 0}
    for key in keys:
        n = len(groups[key])
        if abs(counts['test'] + n - expected_test) < abs(counts['test'] - expected_test):
            split = 'test'
        elif abs(counts['val'] + n - expected_val) < abs(counts['val'] - expected_val):
            split = 'val'
        else:
            split = 'train'
        parts[split].extend(groups[key])
        counts[split] += n
    missing_pairs = all_pairs - {(r['crime_category'],r['crime_subcategory']) for r in parts['train']}
    error = abs(counts['test'] - expected_test) + abs(counts['val'] - expected_val)
    if best is None or (len(missing_pairs), error) < best[0]:
        best = ((len(missing_pairs), error), parts, missing_pairs, attempt)
    if not missing_pairs and error <= 2:
        break

_, parts, missing_pairs, used_attempt = best
assert not missing_pairs, f'Subcategories missing from training: {sorted(missing_pairs)}'
keys_by_split = [set(event_key(r['event']) for r in parts[x]) for x in ('train','val','test')]
assert all(keys_by_split[i].isdisjoint(keys_by_split[j])
           for i in range(3) for j in range(i+1,3))
for split, data in parts.items():
    print(f'{split:5} {len(data):5} examples ({len(data)/len(rows_clean):.2%})')
print('Split attempt:', used_attempt)


## Write and verify splits


In [ ]:
from datetime import datetime, timezone
source_sha = sha256_file(CSV_PATH)
manifest_path = SPLITS_CLEAN / 'manifest.json'
expected_config = {
    'dataset_sha256': source_sha,
    'seed': SEED_CLEAN,
    'clean_method': CLEAN_METHOD,
    'original_count': len(rows_raw)
}

if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    for key, val in expected_config.items():
        assert saved.get(key) == val, f'Existing dataset differs on {key}!'
    expected_files = ['train.jsonl','val.jsonl','test.jsonl','taxonomy.json']
    for name in expected_files:
        p = SPLITS_CLEAN / name
        assert p.exists(), f'Missing {p}: partial splits will not be recreated automatically'
        assert sha256_file(p) == saved['file_sha256'][name], f'Hash mismatch: {p}'
    print('Existing splits verified and preserved:', SPLITS_CLEAN)
else:
    if SPLITS_CLEAN.exists() and any(SPLITS_CLEAN.iterdir()):
        raise FileExistsError(f'{SPLITS_CLEAN} is not empty but lacks manifest.json; refusing to overwrite')
    SPLITS_CLEAN.mkdir(parents=True, exist_ok=True)
    def write_jsonl(path, data):
        with path.open('x', encoding='utf-8') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
    for split, data in parts.items():
        write_jsonl(SPLITS_CLEAN / f'{split}.jsonl', data)
    train_taxonomy = defaultdict(set)
    for r in parts['train']:
        train_taxonomy[r['crime_category']].add(r['crime_subcategory'])
    taxonomy = {k:sorted(v) for k,v in sorted(train_taxonomy.items())}
    (SPLITS_CLEAN/'taxonomy.json').write_text(json.dumps(taxonomy,indent=2,ensure_ascii=False),encoding='utf-8')
    with (SPLITS_CLEAN/'context_changes.csv').open('x',newline='',encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['original_row_id','original','cleaned'])
        writer.writeheader(); writer.writerows(changes)
    hashes = {name: sha256_file(SPLITS_CLEAN/name)
              for name in ('train.jsonl','val.jsonl','test.jsonl','taxonomy.json')}
    manifest = {**expected_config,
                'created_utc': datetime.now(timezone.utc).isoformat(),
                'attempt':used_attempt, 'split_counts':{s:len(parts[s]) for s in parts},
                'changed_contexts':len(changes),
                'remaining_explicit_label_contexts':len(still_mentions),
                'file_sha256':hashes}
    (SPLITS_CLEAN/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    print('Created new experiment:', SPLITS_CLEAN)

print('\nOutput paths (NEW):')
for p in sorted(SPLITS_CLEAN.iterdir()):
    print(' -',p.relative_to(ROOT))


## Next step

Run the paired training notebook with a GPU.
